
# 🧠 Chuleta de **Altair** — Versión DOCENTE

Esta guía está pensada para explicar **Altair** dentro del bloque de **EDA**.  
Altair propone una **gramática de gráficos declarativa** (parecida a ggplot2/Vega‑Lite): describimos el **qué** (datos, codificaciones, agregaciones) y no el **cómo** (píxeles).

**Cómo situarlo frente a Matplotlib/Seaborn:**
- Matplotlib es de **bajo nivel** (control fino, pero más código).
- Seaborn añade **estética estadística** sobre Matplotlib.
- **Altair** prioriza *declaración + estadística + interactividad* con menos código y alta legibilidad.

💡 Úsalo para **storytelling y análisis exploratorio**: filtros, selecciones, agregaciones y composición de vistas.



## ⚙️ Instalación y configuración
💡 **Nota docente**: En Colab/entornos limpios instala paquetes; desactiva el límite de filas para demos con datasets medianos.


In [1]:

#%pip -q install altair==5.* vega_datasets==0.9.* pandas
import altair as alt, pandas as pd
from vega_datasets import data
alt.data_transformers.disable_max_rows()
print("Altair:", alt.__version__)


Altair: 5.5.0



## 📊 Dataset de ejemplo
Usaremos `vega_datasets.cars()` (columnas: `Horsepower`, `Miles_per_Gallon`, `Origin`, `Acceleration`, `Weight_in_lbs`, etc.).

💡 **Nota docente**: Es estable, pequeño y suficiente para enseñar transformaciones básicas e interactividad.


In [2]:

df = data.cars()
df.head()


,Name,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year,Origin
0,chevrolet chevelle malibu,18.0,8,307.0,130.0,3504,12.0,1970-01-01,USA
1,buick skylark 320,15.0,8,350.0,165.0,3693,11.5,1970-01-01,USA
2,plymouth satellite,18.0,8,318.0,150.0,3436,11.0,1970-01-01,USA
3,amc rebel sst,16.0,8,304.0,150.0,3433,12.0,1970-01-01,USA
4,ford torino,17.0,8,302.0,140.0,3449,10.5,1970-01-01,USA



## 🪄 `alt.Chart()` — Comienzo de todo
`alt.Chart(data, title=...)` crea el objeto base. Sobre él encadenamos `mark_*()` y `encode()`.

**Parámetros**: `data`, `title`, `width`, `height` (como propiedades).

💡 **Nota docente**: recalcar separación **datos → marca → codificación**.


In [7]:

base = alt.Chart(df, title="Altair: base + mark + encode")
base.mark_point().encode(x="Horsepower", y="Miles_per_Gallon")


alt.Chart(...)


🧩 **Ejercicio**: Cambia `mark_point()` por `mark_bar()` y prueba con `x='Origin', y='count()'`.



## 🎨 `mark_*()` — Seleccionar el tipo de gráfico
Marcas comunes: `mark_point`, `mark_bar`, `mark_line`, `mark_area`, `mark_circle`.

💡 **Nota docente**: Explica cuándo usar cada una (dispersión, barras agregadas, tendencias).


In [8]:

alt.Chart(df).mark_bar().encode(
    x="Origin",
    y="count()"
).properties(title="Barras por país de origen")


alt.Chart(...)

In [22]:

alt.Chart(df).mark_circle(opacity=0.3,color="tomato").encode(
    x="Horsepower",
    y="Miles_per_Gallon",
    size="Weight_in_lbs",
    tooltip=["Name","Horsepower","Miles_per_Gallon"]
).properties(title="Dispersión con tooltip", width=600, height=400).interactive()


alt.Chart(...)


🧩 **Ejercicio**: Repite el gráfico de puntos con `mark_circle(opacity=0.5)` y añade `size="Acceleration"`.



## 🎯 `encode()` — Mapear columnas a canales visuales
Canales típicos: `x`, `y`, `color`, `size`, `shape`, `tooltip`.

💡 **Nota docente**: introduce sufijos de tipo (`:Q`, `:N`, `:O`, `:T`) cuando haga falta precisión.


In [27]:

alt.Chart(df).mark_point(opacity=0.6).encode(
    x=alt.X("Horsepower:Q", title="HP"),
    y=alt.Y("Miles_per_Gallon:Q", title="MPG"),
    shape="Origin:N",
    size="Acceleration:Q",
    tooltip=["Name","Origin","Horsepower","Miles_per_Gallon","Acceleration"]
).properties(title="Canales: color y tamaño", width=600, height=400).interactive()


alt.Chart(...)


🧩 **Ejercicio**: Cambia `color` por `shape='Origin'` y comenta qué tal legibilidad vs. color.



## 🔁 `transform_*()` — Filtrar, agregar, binning y columnas calculadas
- `transform_filter(expr)`  
- `transform_aggregate(**, groupby=[...])`  
- `transform_bin("binX", field="col", bin=alt.Bin(maxbins=...))`  
- `transform_calculate(nueva="datum.col1 * 10")`

💡 **Nota docente**: resalta que **no** modificamos `df`; la transformación es **dentro** del gráfico.


In [28]:

# Media de HP por Origin (agregación)
alt.Chart(df).transform_aggregate(
    mean_hp="mean(Horsepower)", groupby=["Origin"]
).mark_bar().encode(
    x="Origin:N",
    y=alt.Y("mean_hp:Q", title="HP media"),
    tooltip=["Origin:N","mean_hp:Q"]
).properties(title="Agregación: mean por país", width=500, height=400).interactive()


alt.Chart(...)

In [ ]:

# Histograma de MPG con binning
alt.Chart(df).transform_bin(
    "bin_mpg", field="Miles_per_Gallon", bin=alt.Bin(maxbins=30)
).transform_filter(
    "datum.Origin == 'USA'"
).mark_bar().encode(
    x=alt.X("bin_mpg:Q", title="MPG (binned)"),
    y="count()",
    tooltip=["count()"]
).properties(title="Histograma (binning)")


alt.Chart(...)


🧩 **Ejercicio**: Añade `transform_filter("datum.Origin == 'USA'")` al histograma para comparar con Japón y Europa.



## 🧩 Interactividad — `selection_point` y `selection_interval`
- `selection_point(fields=[...], bind='legend' | widgets)` para selección discreta (categorías).
- `selection_interval()` para selección por rango (ej. brushing).

💡 **Nota docente**: muestra el patrón **add_params → transform_filter**.


In [32]:

sel = alt.selection_point(fields=["Origin"], bind="legend", toggle=True)

alt.Chart(df).mark_circle(size=70).encode(
    x="Horsepower:Q", y="Miles_per_Gallon:Q",
    color=alt.condition(sel, "Origin:N", alt.value("lightgray")),
    tooltip=["Name","Origin","Horsepower","Miles_per_Gallon"]
).add_params(sel).properties(title="Selecciona países en la leyenda").interactive()


alt.Chart(...)

In [40]:

brush = alt.selection_interval(encodings=["x","y"])

points = alt.Chart(df).mark_point().encode(
    x="Horsepower:Q", y="Miles_per_Gallon:Q", color="Origin:N"
).add_params(brush)

hist = alt.Chart(df).mark_bar().encode(
    x=alt.X("Horsepower:Q", bin=True),
    y='count()'
).transform_filter(brush)

(points | hist)


alt.HConcatChart(...)


🧩 **Ejercicio**: Cambia el `hist` para que sea de `Horsepower` y observa cómo cambia la distribución según el *brush*.



## 🧱 `facet()` — Subgráficos por categoría
Divide una vista en paneles por columna/fila.

💡 **Nota docente**: útil para comparar grupos (usa 2–6 facetas; demasiadas pierden legibilidad).


In [39]:

alt.Chart(df).mark_point().encode(
    x="Horsepower:Q", y="Miles_per_Gallon:Q"
).facet(column="Origin:N").properties(title="Facet por origen")


alt.FacetChart(...)


🧩 **Ejercicio**: Cambia a `row="Origin"` y comenta diferencias de layout y lectura.



## 🧩 Composición — *Layering* y *Concatenation*
- **Layer** (`chart1 + chart2`): superponer (ej. puntos + línea de tendencia).
- **Concat** (`chart1 | chart2`, `chart1 & chart2`): lado a lado o apilados.

💡 **Nota docente**: patrón típico para *reference lines* o *trend lines*.


In [50]:

points = alt.Chart(df).mark_circle(opacity=0.4).encode(
    x="Horsepower:Q", y="Miles_per_Gallon:Q",
    tooltip=["Name", "Horsepower", "Miles_per_Gallon"]
)
trend = alt.Chart(df).transform_regression("Horsepower","Miles_per_Gallon").mark_line(color="firebrick").encode(
    x="Horsepower:Q", y="Miles_per_Gallon:Q"
)

rule = alt.Chart(df).mark_rule(color='gray').encode(y=alt.value(30)).properties(title="Linea horizontal en y=30")

(points + trend + rule).properties(width=600, height=400, title="Relación de Horsepower y MPG con línea de tendencia y regla horizontal")


alt.LayerChart(...)


🧩 **Ejercicio**: Añade una línea horizontal en `y=30` usando `mark_rule()` y `encode(y='a_value')` con `transform_calculate`.



## ⚖️ Escalas, ejes y títulos
Controla dominios (`scale`), formatos y títulos de ejes.

💡 **Nota docente**: enfatiza `scale=alt.Scale(zero=False)` para datos que no empiezan en 0.


In [65]:

alt.Chart(df).mark_point().encode(
    x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False, domain=[50, 230]), title="HP"),
    y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False, domain=[10, 50]), title="MPG"),
    color="Origin:N",
    tooltip=["Name","Origin","Horsepower","Miles_per_Gallon"]
).properties(title="Escalas y ejes personalizados")


alt.Chart(...)


🧩 **Ejercicio**: Fija el dominio de `x` a `[50, 200]` y comenta cómo cambia la percepción de la nube de puntos.



## 💾 Guardar y exportar
Exporta a **HTML**, **PNG** o **SVG**. Para PNG/SVG se necesita `altair_saver` o ejecutar en entorno con soporte.

💡 **Nota docente**: Para clase, `chart.save('plot.html')` es suficiente.


In [66]:

plot = alt.Chart(df).mark_bar().encode(x="Origin:N", y="count()").properties(title="Export demo")
plot.save("altair_export_demo.html")
"Guardado: altair_export_demo.html"


'Guardado: altair_export_demo.html'


---
## ✅ Repaso para el docente
- Empezar con `Chart → mark → encode`.
- Mostrar *transformaciones* dentro del gráfico (sin tocar `df`).
- Añadir **interactividad** básica: `selection_point`, `selection_interval`.
- Terminar con **composición** (layer + concat) y **exportación**.

**Errores comunes**: olvidar sufijos de tipo, intentar usar `df.query` en lugar de `transform_filter`, olvidar `.interactive()` cuando hace falta zoom/drag.
